# ProcessGuard: PID lab EDA, training and session evaluation

Notebook for operator-verified PID observations. No malware or workload is executed. Cardiff and synthetic metrics must not be substituted for PID lab evidence. Run after collecting data in a VM; configure TRAIN_SOURCES and EVAL_SOURCES explicitly.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "process_guard").exists() and (ROOT.parent / "process_guard").exists():
    ROOT = ROOT.parent
if not (ROOT / "process_guard").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
sys.path.insert(0, str(ROOT))
RUN = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
TRAIN_SOURCES = [ROOT / "data/observations/collected_samples.csv"]
EVAL_SOURCES = []  # Explicit CSV paths for NEW sessions; never include these in training
MODELS = ROOT / "artifacts/models/pid_live"
REPORTS = ROOT / "artifacts/reports/pid_live"
for source in TRAIN_SOURCES:
    if not source.exists():
        raise FileNotFoundError(f"Collect verified lab observations first: {source}")

## 1. Dataset audit and EDA

Labels are operator supplied. Snapshot rows are correlated. Inspect class balance, independent executable paths, constant features and missing enrichment before training. The pipeline rejects invalid rows rather than silently imputing values.

In [ ]:
from process_guard.pid_visualization import analyze_observations, plot_pid_training_results
from IPython.display import display, Image
EDA = ROOT / "artifacts/reports/pid_eda" / RUN
summary = analyze_observations(TRAIN_SOURCES, EDA)
display(summary)
display(Image(filename=str(EDA / "class_balance.png")))
display(Image(filename=str(EDA / "runtime_features.png")))

## 2. Four models with grouped train/validation/test

Each normalized executable path stays in one partition. At least three independent paths per label are required; copying a binary to another path does not provide independence. Isolation Forest fits benign training rows only. Select the supervised model using validation F1. Test metrics use the same score and 0.65 threshold as live inference. This cell updates PID model/report artifacts; keep previous runs if needed.

In [ ]:
from process_guard.pid_ml import train_labeled_samples
comparison = train_labeled_samples(TRAIN_SOURCES, MODELS, REPORTS)
display(comparison)
figures = plot_pid_training_results(REPORTS)
for figure in figures:
    display(Image(filename=str(figure)))

## 3. Evaluate integrated detectors on NEW sessions

The evaluator recomputes rule, ML and combined scores from features, audits training fingerprints, refuses identical training CSV content or reused process instances where metadata is available, and reports known executable overlap. Known executable paths in new sessions do not demonstrate unseen-executable generalization. Null metrics indicate undefined denominators.

In [ ]:
from process_guard.pid_evaluation import evaluate_observations
if EVAL_SOURCES:
    evaluation = evaluate_observations(EVAL_SOURCES, ROOT / "artifacts/reports/pid_evaluation" / RUN, MODELS, REPORTS)
    display(evaluation)
else:
    print("No held-out sessions configured; no detection/false-positive claim is made.")

## 4. Evidence and report

Restart the desktop/web monitor after training, execute approved scenarios in the VM, retain screenshots, scenario logs, label provenance and monitor benchmark. Complete the VM procedure in docs/LAB_WORKFLOW.md. Unit tests and this notebook do not attest VM isolation or label correctness.

In [ ]:
from scripts.build_project_report import build_report
report_path = ROOT / "docs/BAO_CAO_TONG_HOP.md"
build_report(ROOT, report_path)
print(report_path)